In [1]:
import pandas as pd
from itertools import combinations
from collections import Counter
from google.colab import drive

drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/archive"
prior_path = f"{base_path}/order_products__prior_top1500_user5.csv"

prior_df = pd.read_csv(prior_path)
print(prior_df.shape)
prior_df.head()

Mounted at /content/drive
(18898581, 4)


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,17794,6,1
4,3,33754,1,1


In [2]:
order_baskets = prior_df.groupby("order_id")["product_id"].apply(list)

print("주문 수:", len(order_baskets))
order_baskets.head()

주문 수: 2873678


,product_id
order_id,
2,"[33120, 28985, 9327, 17794]"
3,"[33754, 24838, 21903, 46667, 17461, 32665]"
4,"[46842, 34862, 17616, 25146, 41276]"
5,"[13176, 27966, 23909, 48370, 27360, 6348, 6184..."
9,"[21405, 47890, 11182, 14992, 31506, 23288, 183..."


In [3]:
pair_counter = Counter()

for basket in order_baskets:
    unique_items = sorted(set(basket))  # 같은 주문 내 중복 제거
    for a, b in combinations(unique_items, 2):
        pair_counter[(a, b)] += 1
        pair_counter[(b, a)] += 1  # 양방향 저장

In [4]:
product_counts = prior_df["product_id"].value_counts().to_dict()

In [5]:
rows = []

for (source_product, rec_product), pair_count in pair_counter.items():
    source_count = product_counts.get(source_product, 1)
    score = pair_count / source_count

    rows.append({
        "source_product_id": source_product,
        "recommended_product_id": rec_product,
        "pair_count": pair_count,
        "score": score
    })

cooc_df = pd.DataFrame(rows)
print(cooc_df.shape)
cooc_df.head()

(2056898, 4)


,source_product_id,recommended_product_id,pair_count,score
0,9327,17794,230,0.038943
1,17794,9327,230,0.003297
2,9327,28985,211,0.035726
3,28985,9327,211,0.003282
4,9327,33120,53,0.008974


In [6]:
cooc_df = cooc_df.sort_values(
    ["source_product_id", "score", "pair_count"],
    ascending=[True, False, False]
)

cooc_top10 = cooc_df.groupby("source_product_id").head(10).copy()

cooc_top10["rank"] = cooc_top10.groupby("source_product_id").cumcount() + 1

print(cooc_top10.shape)
cooc_top10.head(20)

(15000, 5)


,source_product_id,recommended_product_id,pair_count,score,rank
41064,34,24852,1643,0.264829,1
24728,34,13176,964,0.155384,2
202572,34,21137,801,0.129110,3
61888,34,47209,663,0.106867,4
24732,34,21903,661,0.106544,5
349998,34,47766,582,0.093810,6
41066,34,27845,521,0.083978,7
245750,34,44632,468,0.075435,8
400634,34,47626,446,0.071889,9
178882,34,27966,410,0.066086,10


In [7]:
products_path = f"{base_path}/products_top1500.csv"
products_df = pd.read_csv(products_path)

cooc_top10_named = cooc_top10.merge(
    products_df[["product_id", "product_name"]],
    left_on="source_product_id",
    right_on="product_id",
    how="left"
).rename(columns={"product_name": "source_product_name"}).drop(columns=["product_id"])

cooc_top10_named = cooc_top10_named.merge(
    products_df[["product_id", "product_name"]],
    left_on="recommended_product_id",
    right_on="product_id",
    how="left"
).rename(columns={"product_name": "recommended_product_name"}).drop(columns=["product_id"])

cooc_top10_named.head(20)

,source_product_id,recommended_product_id,pair_count,score,rank,source_product_name,recommended_product_name
0,34,24852,1643,0.264829,1,Peanut Butter Cereal,Banana
1,34,13176,964,0.155384,2,Peanut Butter Cereal,Bag of Organic Bananas
2,34,21137,801,0.129110,3,Peanut Butter Cereal,Organic Strawberries
3,34,47209,663,0.106867,4,Peanut Butter Cereal,Organic Hass Avocado
4,34,21903,661,0.106544,5,Peanut Butter Cereal,Organic Baby Spinach
5,34,47766,582,0.093810,6,Peanut Butter Cereal,Organic Avocado
6,34,27845,521,0.083978,7,Peanut Butter Cereal,Organic Whole Milk
7,34,44632,468,0.075435,8,Peanut Butter Cereal,Sparkling Water Grapefruit
8,34,47626,446,0.071889,9,Peanut Butter Cereal,Large Lemon
9,34,27966,410,0.066086,10,Peanut Butter Cereal,Organic Raspberries


In [8]:
output_path = f"{base_path}/cooccurrence_top10.csv"
cooc_top10.to_csv(output_path, index=False)

output_named_path = f"{base_path}/cooccurrence_top10_named.csv"
cooc_top10_named.to_csv(output_named_path, index=False)

print("저장 완료:", output_path)
print("저장 완료:", output_named_path)

저장 완료: /content/drive/MyDrive/archive/cooccurrence_top10.csv
저장 완료: /content/drive/MyDrive/archive/cooccurrence_top10_named.csv
